# 05 — Simulação por safra: colheita + 12 meses para vender

O cafeicultor tem o café em mãos na colheita (1º dia útil de julho, pico da colheita do arábica em MG/SP) e um ano para vender. A política LSTM+DP é **rolante**: reavaliada todo mês com a previsão daquela data e o prazo restante (`src/decision/safra.py`). Além de 6 meses, fora do alcance da LSTM, a trajetória é extrapolada de forma conservadora (deriva mantida, incerteza crescendo com √t).

**Sem retreino**: o modelo foi treinado até jun/2022. As safras **2024/25 e 2025/26 são cegas** (teste). As safras 2022/23 e 2023/24 (validação) ampliam a amostra, mas serviram ao early stopping.

Os benchmarks (venda imediata, parcelada em 13 parcelas mensais, média móvel de 3 meses, meta de margem sobre COE, hedge rolado até o fim, oráculo) usam o mesmo horizonte de 12 meses e a mesma contabilidade de custos.

In [3]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
plt.rcParams['figure.figsize'] = (12, 4)

from src.models.dataset import TARGET
from src.models.predict import load_artifacts, residual_sigma
from src.decision.engine import Custos, PASSO
from src.decision.safra import simular_safras, MESES_SAFRA

model, scaler, df = load_artifacts()
sigma = residual_sigma()
custos = Custos(armazenagem_saca_mes=2.5, quebra_pct_mes=0.002, coe_saca=700.0,
                taxa_desconto_aa=0.10, aversao_risco=0.0)  # caso base do anexo: 10 % a.a.; a Selic vigente entra só na sensibilidade (bloco A1)
SAFRAS = [2022, 2023, 2024, 2025]

## 1. Receita líquida por safra (R$/saca, valor presente na colheita)

In [4]:
tab, res = simular_safras(SAFRAS, custos)
tab.to_csv(ROOT / 'reports' / 'simulacao_safras.csv')
est = ['venda_imediata', 'venda_parcelada', 'media_movel_3m', 'meta_margem_coe', 'hedge_ate_o_fim', 'lstm_dp', 'oraculo']
print(tab.round(0).to_string())
ganho = (tab[est].div(tab['venda_imediata'], axis=0) - 1) * 100
print('\nganho vs venda imediata (%):'); print(ganho.round(1).to_string())
print('\nmédia nas safras cegas:'); print(tab.loc[tab.cego, est].mean().round(0).to_string())
print('\nmédia nas 4 safras:'); print(tab[est].mean().round(0).to_string())
print('ganho médio do Agro Visible sobre a venda imediata (média dos ganhos por safra):')
print(f"  2 safras cegas: {ganho.loc[tab.cego, 'lstm_dp'].mean():+.1f} %   4 safras: {ganho['lstm_dp'].mean():+.1f} %")
print('ganho agregado (soma das receitas):')
print(f"  2 safras cegas: {(tab.loc[tab.cego, 'lstm_dp'].sum() / tab.loc[tab.cego, 'venda_imediata'].sum() - 1) * 100:+.1f} %   4 safras: {(tab['lstm_dp'].sum() / tab['venda_imediata'].sum() - 1) * 100:+.1f} %")


          cego  mes_venda_lstm  mes_oraculo  venda_imediata  venda_parcelada  media_movel_3m  meta_margem_coe  hedge_ate_o_fim  lstm_dp  oraculo
2022/23  False               0            0          1359.0           1040.0          1359.0           1359.0           1211.0   1359.0   1359.0
2023/24  False               7           12           824.0            927.0           820.0            884.0            609.0    874.0   1205.0
2024/25   True               0            9          1362.0           1839.0          1362.0           1362.0           1124.0   1362.0   2321.0
2025/26   True               2            2          1807.0           1835.0          2251.0           1807.0           1410.0   2251.0   2251.0

ganho vs venda imediata (%):
         venda_imediata  venda_parcelada  media_movel_3m  meta_margem_coe  hedge_ate_o_fim  lstm_dp  oraculo
2022/23             0.0            -23.4             0.0              0.0            -10.9      0.0      0.0
2023/24             0.0    

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
x = np.arange(len(tab)); w = 0.12
cores = {'venda_imediata': '#8c8c8c', 'venda_parcelada': '#c8a24c', 'media_movel_3m': '#7f9fbf', 'meta_margem_coe': '#b0b0b0',
         'hedge_ate_o_fim': '#5a8f6a', 'lstm_dp': '#a0522d', 'oraculo': '#333333'}
for i, e in enumerate(est):
    ax.bar(x + (i - 3) * w, tab[e], w, label=e, color=cores[e])
ax.set_xticks(x); ax.set_xticklabels([f"{s}{'' if c else ' (val)'}" for s, c in zip(tab.index, tab.cego)])
ax.set_ylabel('R\$/saca (VP na colheita)'); ax.set_title('Receita líquida por safra: colheita em julho, até 12 meses para vender')
ax.legend(ncol=4, fontsize=8); plt.tight_layout()

## 2. Trilha de decisão em cada safra

Preço real (linha), trajetória prevista pela LSTM na colheita (tracejado, ±1σ), ação tomada em cada mês (marcadores), ponto de venda da ferramenta (★) e do oráculo (◆).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, r in zip(axes.ravel(), res):
    cam, tr = r['caminho'], r['trilha']
    ax.plot(cam.index, cam.values, color='sienna', lw=1.2, label='preço real')
    # trajetória prevista na colheita (6 meses) com ±1σ
    t0 = tr.iloc[0]; p0 = t0['preco']
    hs = np.array([0, 21, 63, 126]); mu = np.array([0, t0['y_21d'], t0['y_63d'], t0['y_126d']])
    sd = np.sqrt(np.maximum.accumulate(np.array([0, *sigma.values()]) ** 2))
    dts = [cam.index[min(h, len(cam) - 1)] for h in hs]
    ax.plot(dts, p0 * np.exp(mu), '--', color='steelblue', label='LSTM na colheita')
    ax.fill_between(dts, p0 * np.exp(mu - sd), p0 * np.exp(mu + sd), color='steelblue', alpha=0.15)
    cor = {'VENDER': 'firebrick', 'ARMAZENAR': 'seagreen', 'ARMAZENAR+HEDGE': 'steelblue'}
    for _, row in tr.iterrows():
        ax.scatter(row['data'], row['preco'], color=cor[row['acao']], s=40, zorder=3)
    kv, ko = r['mes_venda'], r['mes_oraculo']
    ax.scatter(cam.index[min(kv * PASSO, len(cam) - 1)], cam.iloc[min(kv * PASSO, len(cam) - 1)], marker='*', s=260, color='gold', edgecolor='k', zorder=4, label=f'venda LSTM+DP (mês {kv})')
    ax.scatter(cam.index[min(ko * PASSO, len(cam) - 1)], cam.iloc[min(ko * PASSO, len(cam) - 1)], marker='D', s=70, color='k', zorder=4, label=f'oráculo (mês {ko})')
    ax.axhline(custos.coe_saca, color='gray', ls=':', lw=0.8)
    ax.set_title(f"Safra {r['safra']} {'(teste cego)' if r['cego'] else '(validação)'} — LSTM+DP R\$ {r['receitas']['lstm_dp']:.0f} × imediata R\$ {r['receitas']['venda_imediata']:.0f}", fontsize=10)
    ax.legend(fontsize=7, loc='best'); ax.set_ylabel('R$/saca')
plt.tight_layout()

## 3. Sensibilidade à aversão a risco e ao mês de colheita

In [7]:
linhas = []
for lam in [0.0, 0.25, 0.5]:
    for mes in [6, 7, 8]:
        c = Custos(**{**custos.__dict__, 'aversao_risco': lam})
        t, _ = simular_safras(SAFRAS, c, mes_colheita=mes)
        g = (t['lstm_dp'] / t['venda_imediata'] - 1) * 100
        linhas.append({'λ': lam, 'mês colheita': mes, 'lstm_dp R$/sc': t['lstm_dp'].mean(), 'ganho vs imediata %': g.mean(),
                       'ganho safras cegas %': g[t.cego].mean(), 'captura oráculo %': (t['lstm_dp'] / t['oraculo'] * 100).mean()})
pd.DataFrame(linhas).set_index(['λ', 'mês colheita']).round(1)

lstm_dp R$/sc  ganho vs imediata %  ganho safras cegas %  \
λ    mês colheita                                                             
0.00 6                    1433.6                 -4.2                  -3.0   
     7                    1461.5                  7.6                  12.3   
     8                    1631.5                 20.2                  36.4   
0.25 6                    1485.0                  0.0                   0.0   
     7                    1461.5                  7.6                  12.3   
     8                    1621.0                 19.4                  36.4   
0.50 6                    1485.0                  0.0                   0.0   
     7                    1361.6                  2.8                   0.0   
     8                    1364.4                  3.7                   2.2   

                   captura oráculo %  
λ    mês colheita                     
0.00 6                          81.0  
     7                          82.8  
     8                          88.9  
0.25 6                          84.6  
     7                          82.8  
     8                          88.1  
0.50 6                          84.6  
     7                          78.8  
     8                          77.7

## 4. Sazonalidade do preço físico

Desvio médio de cada mês em relação à média do ano civil, só em anos completos (2015 até o último ano com 12 meses). É o ganho estrutural que armazenar da colheita (junho–setembro) até o verão captura, independentemente da previsão. Sai em `reports/experimentos/sazonalidade_mensal.csv` e `reports/figures/13_sazonalidade_mensal.png`.


In [ ]:
P = df[TARGET].loc['2015':]
completos = [a for a, g in P.groupby(P.index.year) if g.index.month.nunique() == 12]
P = P[P.index.year.isin(completos)]
desvio = (P / P.groupby(P.index.year).transform('mean') - 1) * 100
saz = desvio.groupby(desvio.index.month).agg(['mean', 'median', 'std', 'count'])
saz.index.name = 'mes'
saz.columns = ['desvio_medio_%', 'desvio_mediano_%', 'desvio_dp_%', 'n_dias']
saz['anos'] = f'{min(completos)}-{max(completos)}'
saz.round(2).to_csv(ROOT / 'reports' / 'experimentos' / 'sazonalidade_mensal.csv')
print(f'anos completos: {min(completos)} a {max(completos)} ({len(completos)} anos)')
print(saz.round(1).to_string())

meses = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']
meses = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']
fig, ax = plt.subplots(figsize=(8, 3.6))
x = range(1, 13); w = 0.38
ax.bar([i - w / 2 for i in x], saz['desvio_mediano_%'], width=w, color='#4F6B3B', label='mediana entre anos')
ax.bar([i + w / 2 for i in x], saz['desvio_medio_%'], width=w, color='#c9b48a', label='média entre anos')
ax.axhline(0, color='gray', lw=0.8); ax.set_xticks(list(x)); ax.set_xticklabels(meses)
ax.set_ylabel('desvio da média do ano (%)')
ax.set_title(f'Sazonalidade do indicador CEPEA arábica, {min(completos)}–{max(completos)}')
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(ROOT / 'reports' / 'figures' / '13_sazonalidade_mensal.png', dpi=300, facecolor='white'); plt.show()


## Leitura

- A simulação por safra é o cenário realista do produtor: uma decisão na colheita, com reavaliação mensal, e não vinte decisões independentes como no notebook 04.
- Com 4 safras (2 cegas) a amostra é pequena; a tabela de sensibilidade ao mês de colheita (junho/julho/agosto) triplica os pontos de partida sem mudar o regime.
- Ver `reports/simulacao_safras.csv` para os números por safra.